#### 1. Data Loading & Date Parsing

-imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os


-Load csv 

In [2]:
df = pd.read_csv("../data/tanzania.csv")


-Add country identity

In [3]:
df["Country"] = "Tanzania"

-Convert YEAR and DOY to Datetime

In [4]:
df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')

-Extract Month for Seasonal Analysis

In [5]:
df['Month'] = df['Date'].dt.month

-Reordering columns for better readability


In [7]:
cols = ['Date', 'Country', 'Month', 'YEAR', 'DOY', 'T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
df= df[cols + [c for c in df.columns if c not in cols]]

print("First 5 rows of processed Tanzania data:")
df.head()

First 5 rows of processed Tanzania data:


,Date,Country,Month,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,T2M_RANGE,RH2M,WS2M,WS2M_MAX,PS,QV2M
0,2015-01-01,Tanzania,1,2015,1,27.56,29.52,26.22,7.24,3.30,80.97,4.68,6.01,100.52,18.61
1,2015-01-02,Tanzania,1,2015,2,27.59,29.99,25.92,3.40,4.07,79.63,4.48,5.28,100.52,18.31
2,2015-01-03,Tanzania,1,2015,3,27.47,29.29,26.25,7.17,3.04,80.02,4.91,5.99,100.56,18.30
3,2015-01-04,Tanzania,1,2015,4,27.28,29.17,25.96,16.07,3.21,81.78,4.88,6.07,100.47,18.52
4,2015-01-05,Tanzania,1,2015,5,26.68,27.83,25.84,18.83,1.99,82.99,4.17,5.98,100.43,18.16


##### Analytical Reasoning
Analytical Note: This time-series transformation enables us to distinguish between the Unimodal rainfall (one long season) in the south/west and the Bimodal rainfall in the north/east. Tracking these trends over 11 years helps determine if the warming Indian Ocean is causing "Rainfall Variability," which impacts hydropower generation and food security across the Great Lakes region.

#### 2. Summary Statistics & Missing-Value 


-Replace NASA sentinel values


* NASA POWER uses -999 as a sentinel value for missing data;
these were replaced with NaN to prevent statistical bias.

In [ ]:
df.replace(-999, np.nan, inplace=True)

,Date,Country,Month,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,T2M_RANGE,RH2M,WS2M,WS2M_MAX,PS,QV2M
0,2015-01-01,Tanzania,1,2015,1,27.56,29.52,26.22,7.24,3.30,80.97,4.68,6.01,100.52,18.61
1,2015-01-02,Tanzania,1,2015,2,27.59,29.99,25.92,3.40,4.07,79.63,4.48,5.28,100.52,18.31
2,2015-01-03,Tanzania,1,2015,3,27.47,29.29,26.25,7.17,3.04,80.02,4.91,5.99,100.56,18.30
3,2015-01-04,Tanzania,1,2015,4,27.28,29.17,25.96,16.07,3.21,81.78,4.88,6.07,100.47,18.52
4,2015-01-05,Tanzania,1,2015,5,26.68,27.83,25.84,18.83,1.99,82.99,4.17,5.98,100.43,18.16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4103,2026-03-27,Tanzania,3,2026,86,27.63,30.65,25.66,2.96,4.99,77.48,1.57,2.40,100.73,17.72
4104,2026-03-28,Tanzania,3,2026,87,27.51,31.23,24.84,1.65,6.39,77.72,1.36,1.79,100.61,17.59
4105,2026-03-29,Tanzania,3,2026,88,27.74,31.21,25.33,1.28,5.88,77.35,1.53,2.01,100.42,17.81
4106,2026-03-30,Tanzania,3,2026,89,27.83,31.29,25.31,0.92,5.98,76.50,1.64,2.15,100.43,17.72


-Duplicate Check


In [9]:
duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")
df = df.drop_duplicates()

Duplicate rows found: 0


-Missing Value 


In [10]:
missing = df.isna().sum()
missing_percent = (missing/ len(df)) * 100

print("\nMissing Value Percentages per Column:")
print(missing_percent[missing_percent > 0])


Missing Value Percentages per Column:
Series([], dtype: float64)


-Generate Summary Statistics
* We focus on temperature and precipitation for climate trends


In [11]:
summary_stats= df[['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M']].describe()
summary_stats

,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M
count,4108.000000,4108.000000,4108.00000,4108.000000,4108.000000
mean,26.802422,29.163493,25.03813,3.740256,77.128038
std,1.325388,1.371155,1.53544,8.003947,5.070501
min,22.680000,25.410000,20.03000,0.000000,60.060000
25%,25.670000,28.090000,23.71000,0.110000,73.167500
50%,26.990000,29.080000,25.37500,0.640000,76.585000
75%,27.900000,30.170000,26.31000,3.790000,81.100000
max,29.970000,33.930000,28.01000,122.650000,91.100000


##### Statistical Interpretation

* Temperature: We look for the influence of the Indian Ocean. A high T2M_MIN (warm nights) is a sign of coastal humidity trapping heat. This is a key metric for understanding malaria transmission risks, as warmer nights allow mosquitoes to remain active longer.

* Missing Data: Tanzania’s hydropower depends on consistent rainfall in the highlands. If more than 5% of PRECTOTCORR or WS2M (Wind Speed) is missing, we cannot accurately model evapotranspiration rates, which are critical for managing the country’s energy and water reservoirs.

#### 3. Outlier Detection & Basic Cleaning

-Define columns for outlier analysis

In [12]:
outlier_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

-Compute Z-scores


In [13]:
z_scores = np.abs(stats.zscore(df[outlier_cols].dropna()))

-Flag rows where |Z| > 3
* Note: We re-index to match the original dataframe because zscore drops NaNs


In [15]:
outliers = (np.abs(stats.zscore(df[outlier_cols], nan_policy='omit')) > 3)
outlier_counts = outliers.sum()

print("--- Outlier Count (|Z| > 3) ---")
print(outlier_counts)

--- Outlier Count (|Z| > 3) ---
102


##### Outlier Decision & Reasoning
* Decision on Outliers: > * Temperature/Humidity: We will retain these outliers. In the context of COP32, values with $|Z| > 3$ often represent extreme weather events (heatwaves or cold snaps) rather than data errors.
* Precipitation: We will retain these as well, as they likely represent extreme rainfall events or flash floods, which are critical for "Loss and Damage" policy discussions.
* Action: No rows will be dropped based on Z-score alone, as they provide the "evidence-grade" anomalies required for negotiation.

##### Handling Missing Values & Final Cleaning

-Drop rows with more than 30% missing values

In [16]:
threshold = 0.3 * len(df.columns)
initial_shape = df.shape
df = df.dropna(thresh=int(len(df.columns) - threshold))
dropped_rows = initial_shape[0] - df.shape[0]

-Forward-fill remaining missing weather values



In [17]:
df = df.sort_values('Date').ffill()

print(f"Rows dropped (too many missing values): {dropped_rows}")
print(f"Remaining missing values: {df.isna().sum().sum()}")


Rows dropped (too many missing values): 0
Remaining missing values: 0


-Exporting Clean Data


In [18]:
# Create data directory if it doesn't exist

if not os.path.exists('../data'):
    os.makedirs('../data')

# Export to CSV
output_path = '../data/tanzania_clean.csv'
df.to_csv(output_path, index=False)

print(f"✅ Cleaned data exported to {output_path}")

✅ Cleaned data exported to ../data/tanzania_clean.csv


##### Final Documentation
* Forward-Fill (ffill): Coastal and Great Lakes weather transitions are physically consistent from day to day. ffill preserves the relationship between wind speed (WS2M) and temperature, which is essential for accurate evapotranspiration estimates.

* Data Integrity: Exporting tanzania_clean.csv ensures that our Cross-Country Comparison can accurately weigh the impact of Indian Ocean warming on East African energy and water security.